In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
sys.path.append(str(project_root))

### Market-readiness design

The model distinguishes between:

1. **Minimum feasibility conditions**, which determine whether a country can be
   assessed as currently investable for mini-grid development; and

2. **Readiness-strength factors**, which differentiate feasible markets based on
   the quality of their enabling environment.

High electricity need alone does not imply near-term investment feasibility if
the regulatory pathway, tariff framework, or protection against grid arrival is unclear.


### Evidence rules for market readiness

Each criterion will be assessed using a consistent, verifiable source and recorded
as `Yes`, `No`, or `Unclear`.

| Criterion | Operational definition | Proposed treatment |
|---|---|---|
| Mini-grid / off-grid regulation | A law, regulation, or official framework explicitly governs mini-grid or off-grid electricity development. | Feasibility gate |
| Licensing pathway | An official process specifies whether and how a developer obtains a licence, permit, registration, or exemption. | Feasibility gate |
| Tariff framework | Developers may charge tariffs under a defined approval, cap, negotiated, or cost-reflective process. | Feasibility gate |
| Grid-arrival provisions | Official rules address what happens when the main grid reaches a mini-grid service area, such as compensation, asset transfer, or interconnection. | Readiness-strength factor |
| National DRE support | A current national electrification, energy, or mini-grid plan explicitly recognises decentralised renewable energy or mini-grids. | Readiness-strength factor |

A country will be classified as **currently feasible** only where the first three
conditions are evidenced. A missing source is recorded as `Unclear`, not assumed
to mean `No`.

### Candidate cross-country source: RISE

The Regulatory Indicators for Sustainable Energy (RISE), developed by the World Bank
and ESMAP, will be evaluated as the primary cross-country source for market-readiness
evidence.

RISE is suitable for initial screening because it provides comparable country-level
indicators on sustainable-energy policy and regulation, including a mini-grid topic.

The assessment will test whether the available RISE mini-grid data can consistently
support the following criteria across the 48-country SSA sample:

- mini-grid / off-grid regulation;
- licensing pathway;
- tariff framework;
- grid-arrival provisions; and
- national support for decentralised renewable energy.

RISE results will be used for initial classification only. Where a country is
shortlisted, its current official regulations and electrification plans must be
checked directly, because regulatory frameworks can change after the RISE assessment
year.

In [2]:
# %%
from src.load_data import load_rise_data
import numpy as np

rise_raw = load_rise_data(
    project_root / "data" / "raw" / "market readiness" / "rise-data-score-minigrid.xls"
)
rise_raw.head()



,Category,Score
0,Afghanistan,46
1,Albania,100
2,Algeria,100
3,Angola,83
4,Argentina,100


In [3]:
# name-matching check. Load your 48 SSA countries and list every one whose World Bank name doesn't appear in RISE
from src.pipeline import prepare_investment_dataset


ssa_df = prepare_investment_dataset()

unmatched = ssa_df.loc[
    ~ssa_df["Country Name"].isin(rise_raw["Category"]),
    ["Country Name", "Country Code"],
]
unmatched

,Country Name,Country Code
4,Botswana,BWA
6,Cote d'Ivoire,CIV
10,Comoros,COM
11,Cabo Verde,CPV
14,Gabon,GAB
17,"Gambia, The",GMB
18,Guinea-Bissau,GNB
19,Equatorial Guinea,GNQ
22,Lesotho,LSO
27,Mauritius,MUS


In [4]:
rise_name_fixes = {
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Somalia": "Somalia, Fed. Rep.",
}

rise_df = rise_raw.rename(columns={"Category": "Country Name", "Score": "RISE_Minigrid_Score"})
rise_df["Country Name"] = rise_df["Country Name"].replace(rise_name_fixes)

## v4 notes: RISE mini-grid data

**Loading**
- `rise-data-score-minigrid.xls` is an HTML table saved as .xls, not a real Excel file.
  Read it with `pd.read_html` (needs `lxml`), not `pd.read_excel`.
- Loader lives in `src/load_data.py` as `load_rise_data()`.
- The file has country names but no country codes, and no year column (2023 edition, per the RISE site).

**Name matching**
- Only 2 SSA names differed: Côte d'Ivoire (accent) and Somalia (vs "Somalia, Fed. Rep.").
  Fixed with an explicit mapping, no fuzzy matching.
- 13 SSA countries are not in the RISE mini-grid data: Botswana, Comoros, Cabo Verde,
  Gabon, Gambia, Guinea-Bissau, Equatorial Guinea, Lesotho, Mauritius, Namibia,
  São Tomé and Príncipe, Eswatini, Seychelles.
- Coverage: 35/48 (~73%), below the 90% rule set in v3.

**Decision: RISE stays out of the weighted score**
1. Consistency: v3 rejected other indicators for coverage below 90%.
2. Missing countries aren't random (mostly small/high-access). Filling with 0 or the
   average invents data; dropping them breaks comparability with v1/v2.
3. A weighted sum lets high need hide low readiness (e.g. South Sudan: RISE 5/100).
   Readiness is a gate, not something need can offset.
4. Need vs readiness as two dimensions gives a clearer decision view (quadrants).

Result: RISE becomes a separate readiness column; the 13 gaps are labelled
"Not assessed by RISE", not NaN. Trade-off: no single top-10 list.

In [5]:
rise_ssa = rise_df.merge(
    ssa_df[["Country Name", "Country Code"]],
    on="Country Name",
    how="right",
)
rise_ssa.shape

(48, 3)

In [6]:
rise_ssa["RISE_Minigrid_Score"].notna().sum()

np.int64(35)

In [7]:
rise_ssa["RISE_Status"] = np.where(
    rise_ssa["RISE_Minigrid_Score"].notna(),
    "Assessed",
    "Not assessed by RISE",
)

rise_ssa["RISE_Status"].value_counts()

RISE_Status
Assessed                35
Not assessed by RISE    13
Name: count, dtype: int64

In [8]:
ssa_df.shape
ssa_df["GDP_Score"].isna().sum()

np.int64(2)

In [9]:
rise_ssa.columns

Index(['Country Name', 'RISE_Minigrid_Score', 'Country Code', 'RISE_Status'], dtype='str')

In [10]:
from src.clean_data import merge_indicator

rise_subset = rise_ssa[
    [
        "Country Code",
        "RISE_Minigrid_Score",
        "RISE_Status"
    ]
].copy()

readiness_df = merge_indicator(ssa_df, rise_subset)

readiness_df.shape

(48, 89)

In [11]:
readiness_df["RISE_Status"].value_counts()

RISE_Status
Assessed                35
Not assessed by RISE    13
Name: count, dtype: int64

In [12]:
from src.scoring import calculate_investment_opportunity_score

readiness_df = calculate_investment_opportunity_score(
    readiness_df,
    access_weight=0.25,
    improvement_weight=0.25,
    population_weight=0.25,
    gdp_weight=0.25,
    output_column="Score_With_Improvement",
)

readiness_df = calculate_investment_opportunity_score(
    readiness_df,
    access_weight=1/3,
    improvement_weight=0,
    population_weight=1/3,
    gdp_weight=1/3,
    output_column="Score_Without_Improvement",
)

comparison = readiness_df[
    ["Country Name", "Score_With_Improvement", "Score_Without_Improvement"]
].copy()

comparison["Rank_With"] = comparison["Score_With_Improvement"].rank(ascending=False, method="min")
comparison["Rank_Without"] = comparison["Score_Without_Improvement"].rank(ascending=False, method="min")
comparison["Rank_Change"] = comparison["Rank_With"] - comparison["Rank_Without"]

comparison.sort_values("Rank_Without").head(15)

,Country Name,Score_With_Improvement,Score_Without_Improvement,Rank_With,Rank_Without,Rank_Change
8,"Congo, Dem. Rep.",0.549237,0.660783,1.0,1.0,0.0
31,Nigeria,0.522629,0.637104,3.0,2.0,1.0
13,Ethiopia,0.526311,0.485672,2.0,3.0,-1.0
41,Chad,0.384563,0.478090,12.0,4.0,8.0
3,Burkina Faso,0.350835,0.441231,23.0,5.0,18.0
28,Malawi,0.354229,0.436907,19.0,6.0,13.0
43,Tanzania,0.469989,0.435649,4.0,7.0,-3.0
30,Niger,0.356284,0.434485,18.0,8.0,10.0
0,Angola,0.434530,0.430406,8.0,9.0,-1.0
25,Mozambique,0.351277,0.377662,22.0,10.0,12.0


In [13]:
comparison.reindex(comparison["Rank_Change"].abs().sort_values(ascending=False).index).head(8)

,Country Name,Score_With_Improvement,Score_Without_Improvement,Rank_With,Rank_Without,Rank_Change
32,Rwanda,0.444037,0.258716,7.0,40.0,-33.0
22,Lesotho,0.361042,0.255725,15.0,42.0,-27.0
20,Kenya,0.448243,0.293085,6.0,32.0,-26.0
19,Equatorial Guinea,0.292945,0.375844,34.0,12.0,22.0
3,Burkina Faso,0.350835,0.441231,23.0,5.0,18.0
2,Benin,0.363859,0.308154,14.0,30.0,-16.0
47,Zimbabwe,0.415145,0.326388,9.0,25.0,-16.0
40,Seychelles,0.256084,0.333333,40.0,24.0,16.0


In [14]:
readiness_df.loc[
    readiness_df["Country Name"].isin(["Burkina Faso", "Chad", "Kenya", "Rwanda"]),
    ["Country Name", "2014", "2023", "Improvement", "Access_Score", "Population_Score", "GDP_Score"],
]

,Country Name,2014,2023,Improvement,Access_Score,Population_Score,GDP_Score
3,Burkina Faso,19.2,21.7,2.5,0.827696,0.203907,0.292090
20,Kenya,36.0,76.2,40.2,0.251586,0.148958,0.478710
32,Rwanda,19.8,63.9,44.1,0.381607,0.056974,0.337568
41,Chad,8.4,12.0,3.6,0.930233,0.192276,0.311762


In [ ]:
from src.load_data import load_rise_data

# Use the actual RISE mini-grid file used elsewhere in this notebook
rise_sub_indicators = load_rise_data(
    project_root / "data" / "raw" / "market readiness" / "rise-data-score-minigrid.xls"
)

rise_sub_indicators.head()

ImportError: `Import html5lib` failed.  Use pip or conda to install the html5lib package.